# 05.1 Neutering Cycle Model Benchmark

This notebook is a benchmark variant of `05_neutering_cycle.ipynb`.

It is used to rerun the S02 iterative neutering cycle with alternative N rewriter models, especially stronger N candidates.

For clean benchmark presets, J1 is intentionally kept fixed as `qwen/qwen3.5-397b-a17b` so the benchmark isolates the effect of the N rewriter model.

The `gpt54__sonnet46` preset is a diagnostic exception: it changes J1 to Claude Sonnet 4.6 only to compare the same GPT rewriter under a different in-loop judge.

It must not be treated as the canonical S02 run unless explicitly promoted after review.

Outputs are written under `data/runs/e06_cycle_model_bench/<RUN_LABEL>/` to avoid overwriting accepted S02 artifacts.


# 05 — E06 Neutering Cycle

Multi-iteration neutralization cycle for `calm_2021` and `shock_war`, reusing the notebook 04 single-iteration contracts.

## Config

In [1]:
import json
import os
import re
import sqlite3
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
load_dotenv(PROJECT_ROOT / ".env")

DB_PATH = "data/db/blind_prophet.db"
CALM_2021_RAW_SOURCE = Path("data/runs/e06_smoke/calm_2021_hardened_p03/raw_summary.md")

BENCHMARK_MODE = True

MODEL_PRESETS = {
    "deepseek_v4_flash__qwen397b": {
        "MODEL_N": "deepseek/deepseek-v4-flash",
        "MODEL_J1": "qwen/qwen3.5-397b-a17b",
        "description": "Current accepted S02 baseline: cheap/fast N + large Qwen J1",
    },
    "gpt54__qwen397b": {
        "MODEL_N": "openai/gpt-5.4",
        "MODEL_J1": "qwen/qwen3.5-397b-a17b",
        "description": "Strong OpenAI N candidate + same large Qwen J1; clean N-only benchmark",
    },
    "sonnet46__qwen397b": {
        "MODEL_N": "anthropic/claude-sonnet-4.6",
        "MODEL_J1": "qwen/qwen3.5-397b-a17b",
        "description": "Strong Claude N candidate + same large Qwen J1; clean N-only benchmark",
    },
    "gpt54__sonnet46": {
        "MODEL_N": "openai/gpt-5.4",
        "MODEL_J1": "anthropic/claude-sonnet-4.6",
        "description": "Strong OpenAI N + strong Claude J1; diagnostic cross-judge comparison against gpt54__qwen397b",
    },
}

ACTIVE_PRESET = "sonnet46__qwen397b"
POINT_FILTER = ["shock_war"]

# POINT_FILTER = None


RUN_LABEL = ACTIVE_PRESET
MODEL_N = MODEL_PRESETS[ACTIVE_PRESET]["MODEL_N"]
MODEL_J1 = MODEL_PRESETS[ACTIVE_PRESET]["MODEL_J1"]
MODEL_PRESET_DESCRIPTION = MODEL_PRESETS[ACTIVE_PRESET]["description"]

OUT_BASE_DIR = Path("data/runs/e06_cycle_model_bench")
OUT_ROOT = OUT_BASE_DIR / RUN_LABEL
# Examples:
# POINT_FILTER = ["shock_war"]
# POINT_FILTER = ["calm_2021", "shock_war"]

POINTS = [
    {
        "point_label": "calm_2021",
        "run_date": "2021-10-20",
        "out_dir": OUT_ROOT / "calm_2021",
    },
    {
        "point_label": "shock_war",
        "run_date": "2022-03-25",
        "out_dir": OUT_ROOT / "shock_war",
    },
]

ACTIVE_POINTS = [
    p for p in POINTS
    if POINT_FILTER is None or p["point_label"] in POINT_FILTER
]

N_MAX = 3
Y_FLOOR = 0.70
ROLLBACK_ON_HARD_RESIDUAL = True
MANUAL_STOP = False

# If OpenRouter availability changes, adjust model IDs in MODEL_PRESETS only.

TEMPERATURE_J1 = 0.1
TEMPERATURE_N = 0.3

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"


## Helpers

In [2]:
def load_summary(db_path: str, run_date: str) -> str:
    path = Path(db_path)
    if not path.exists():
        raise FileNotFoundError(f"SQLite DB not found: {path}")

    with sqlite3.connect(path) as conn:
        row = conn.execute(
            "SELECT summary FROM summaries WHERE run_date = ?",
            (run_date,),
        ).fetchone()

    if row is None or not row[0]:
        raise LookupError(f"No summary found in table 'summaries' for run_date={run_date!r}")
    return row[0]


def call_llm(model: str, system_prompt: str, user_prompt: str, temperature: float) -> str:
    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is not set in the environment")

    client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=api_key)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
    )
    content = response.choices[0].message.content
    if content is None:
        raise RuntimeError(f"Model {model} returned empty content")
    return content.strip()


def extract_json(raw_text: str) -> dict:
    text = raw_text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        if lines and lines[0].strip().startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines).strip()

    try:
        parsed = json.loads(text)
    except json.JSONDecodeError as exc:
        preview = text[:500]
        raise ValueError(f"Could not parse strict JSON after optional fence stripping: {exc}\nPreview:\n{preview}") from exc

    if not isinstance(parsed, dict):
        raise ValueError(f"Expected a JSON object, got {type(parsed).__name__}")
    return parsed


Q1_EVIDENCE_SYSTEM = """
You are J1, an in-loop leakage evidence judge for a Russian macroeconomic summary.

Task: identify phrases that could let a model infer the true historical period of the summary.
Return strict JSON only. Do not include markdown, commentary, or extra keys.

Output schema:
{
  "identifiers": [
    {
      "id": "id_001",
      "class": "number | event | time_anchor",
      "pattern_description": "...",
      "anchor_phrase": "...",
      "suggested_strategy": "..."
    }
  ]
}

Identifier classes:
- number: exact numeric coordinates and numeric constellations that identify a cycle phase, including distinctive levels, rates, prices, quantities, or combinations of them.
- event: named events; institutional or public symbolic statements; macro-configuration fingerprints; and combinations of facts that become identifying together even if each individual fact is not unique.
- time_anchor: explicit dates; month/year references; relative references to historically unique comparisons; seasonal phrases that point to a specific calendar window; chronology labels, holidays, or survey-window clues.

Q1 must detect compound fingerprints, including macro-configuration fingerprints:
- commodity shock + ruble strengthening + fiscal surplus.
- inflation spike + administrative price control.
- labor shortage + delivery wage spike.
- fiscal health spending + inflation-linked benefits.
- energy price shock + imported inflation + expected monetary tightening.
- high inflation + expected policy tightening + strong ruble.
- pandemic/health-crisis fiscal spending.

Q1 must detect symbolic-statement fingerprints, including:
- memorable public statements by central bank leadership.
- presidential / ministerial verbal signals that are unusually period-specific.
- statements about reserve currencies, savings currency, stagflation, or sanctions when they identify a historical episode.
- symbolic central-bank/government statements that are searchable or episode-specific.

Q1 must detect unique fact combinations:
- combinations that are individually generic but jointly identify the period.
- named companies, banks, officials, agencies, brands, or countries when not needed for preserving forecast signal.

If several individually generic facts jointly form a recognizable historical fingerprint, flag the combination as an `event` identifier. Do not only flag isolated dates or numbers.

Strict leakage rule: you must NOT leak the true period in your answer.
- No years.
- No month names.
- No exact dates.
- No unique historical event names.
- No euphemisms or aliases that identify the exact episode.

pattern_description must describe the leakage mechanism, not the historical episode.
pattern_description must not contain phrases that would allow direct search-engine identification of the period.
Do not write phrases like "post-pandemic reopening", "initial sanctions shock", "commodity supercycle of <period>", or any equivalent that names the episode indirectly.
Describe only leakage classes and mechanisms. The anchor_phrase may quote or minimally normalize the source phrase, but if the source phrase contains a forbidden exact period marker, mask it with a generic placeholder.
Use stable ids id_001, id_002, ... .
""".strip()


Q3_SIGNALS_SYSTEM = """
You are J1, an in-loop forecast-signal judge for a Russian macroeconomic summary.

Task: extract only forecast-relevant inflation expectation signals that are recoverable from the summary.
Return strict JSON only. Do not include markdown, commentary, or extra keys.

Output schema:
{
  "signals": [
    {
      "id": "sig_001",
      "category": "monetary_policy | supply_shock_food | supply_shock_energy | currency_pressure | fiscal_expansion | demand_shift | regulated_prices | inflation_print",
      "direction": "up | down | mixed | neutral",
      "strength": 1,
      "anchor_phrase": "..."
    }
  ],
  "summary_direction": "up | down | mixed | neutral",
  "summary_confidence": 0.0
}

Use the closed category list only:
- monetary_policy
- supply_shock_food
- supply_shock_energy
- currency_pressure
- fiscal_expansion
- demand_shift
- regulated_prices
- inflation_print

No extra categories are allowed.
direction must be exactly one of: up, down, mixed, neutral.

Direction conventions:
- monetary_policy up means tightening pressure: key-rate increase, expected key-rate increase, higher policy-rate path, or tighter lending / macroprudential stance if inflationary pressure is the reason.
- monetary_policy down means easing pressure: key-rate decrease, expected key-rate decrease, softer policy path, or credit easing.
- monetary_policy neutral means no directional monetary-policy signal.
- monetary_policy mixed means both tightening and easing signals are present.
- regulated_prices down means administrative price containment, tariffs frozen, import-duty relief, or explicit anti-price measures.
- regulated_prices up means regulated/tariff prices increase.
- regulated_prices mixed means both containment and increases appear.

strength must be an integer from 1 to 3, where 1=weak, 2=medium, 3=strong.
Extract only signals actually present in the text; do not infer hidden signals from general macro knowledge.
Do not reward the rewriter for adding new signals.
anchor_phrase must be short and traceable to the text, copied or minimally normalized from the summary.
summary_confidence must be a number from 0.0 to 1.0.
Use stable ids sig_001, sig_002, ... .
""".strip()


N_REWRITER_SYSTEM = """
You are N, a neutralizing rewriter for a Russian macroeconomic summary.

Task: rewrite the previous summary to reduce period-identifying markers while preserving forecast-relevant inflation expectation signals.
Return plain text only. Do not return JSON. Do not include markdown fences or explanations.

Invariants:
- Length must stay within +/-20% of the input summary.
- Preserve event order.
- Mask identifier anchors according to the suggested strategies from J1 Q1-evidence.
- Preserve Q3 signals so that category, direction, and strength +/-1 remain recoverable.
- Levels and numeric coordinates should be generalized.
- Deltas and changes may be preserved when they carry forecast signal.
- Absolute dates should be converted to relative references or removed.
- Events should be rewritten through mechanism and role, without names, places, or dates when those identify the period.
- Do not add new facts.
- Do not turn the text into a generic macro summary; keep the causal structure and sector detail.

Residual fingerprint neutralization:
- Neutralize broad recognizable configurations, not only explicit identifiers.
- The output must not contain month names.
- The output must not contain years.
- The output must not contain named commodity benchmarks when they are not needed for signal preservation: Brent, WTI, Urals.
- The output must not contain unique brands, banks, or country names when they act as period anchors, including specific retail chains, specific global banks, and specific foreign countries in price-record examples.
- Avoid broad but period-identifying labels: "сырьевой суперцикл", "стагфляционный сценарий", "голландская болезнь", "годовой минимум", "исторический максимум", "за три квартала".
- Avoid memorable symbolic-statement anchors unless they are essential for forecast signal.
- Do not preserve a unique combination of facts if the combination itself identifies the period. When several individually generic facts jointly form a historical fingerprint, generalize one or more of them while preserving macro direction and forecast-relevant signal.
- If a residual blacklist term appears in the output, the rewrite should be considered failed.
- Replace "commodity supercycle", "сырьевой суперцикл", and equivalent labels with a neutral mechanism phrase such as "global commodity-price pressure".
- Replace "historical record", "absolute maximum", "unseen since X", and equivalents with less identifying qualitative intensity, such as "very high", "extreme", or "materially elevated", unless the record itself is forecast-relevant.
- Replace "pandemic", "COVID", and similar episode anchors with "health-related fiscal pressure" or "public-health spending pressure" when exact event identity is not required.
- Replace named institutions, companies, officials, banks, and agencies with functional roles if their identity is not necessary for the signal.
- Rewrite symbolic public statements as generic confidence/signaling mechanisms.
- Avoid preserving search-identifiable phrases verbatim.
- Do not introduce new dates, names, or historically unique labels.

The neutralization scope covers the entire text including structural sections such as «Связи» (connections) and «Темы узла» (node themes). These sections are not metadata — neutralize them on the same terms as the narrative body. Specifically: remove named trigger descriptions (e.g. «рост цен на газ в Европе стал ключевым триггером»), generalize thematic labels (e.g. «стагфляция» → «риск одновременно высокой инфляции и слабого роста»), and apply the same level/delta replacement strategy to any numbers in these sections.

Do not over-neutralize:
- Preserve direction of inflation pressure.
- Preserve relative severity of food, energy, currency, fiscal, labor-market, and monetary-policy signals.
- Preserve causal links between commodity prices, imported inflation, food prices, incomes, fiscal measures, and monetary policy.
- Preserve important deltas and rates of change when they are predictive rather than identifying.

Before finalizing, silently check whether a judge could still identify the period from a phrase search, a unique combination of macro facts, a symbolic statement, named public actors, or pandemic/health-crisis anchors. If yes, rewrite those parts more generically while preserving Q3 signals.
""".strip()

def q3_preservation_score(baseline: dict, candidate: dict) -> float:
    baseline_signals = baseline.get("signals", [])
    candidate_signals = candidate.get("signals", [])
    if not baseline_signals:
        return 1.0

    candidate_by_category = {}
    for signal in candidate_signals:
        candidate_by_category.setdefault(signal.get("category"), []).append(signal)

    matched = []
    direction_matches = 0
    strength_consistent = 0

    for base_signal in baseline_signals:
        category = base_signal.get("category")
        candidates = candidate_by_category.get(category, [])
        if not candidates:
            continue
        cand_signal = candidates.pop(0)
        matched.append((base_signal, cand_signal))

        if cand_signal.get("direction") == base_signal.get("direction"):
            direction_matches += 1

        try:
            base_strength = int(base_signal.get("strength"))
            cand_strength = int(cand_signal.get("strength"))
        except (TypeError, ValueError):
            continue
        if abs(base_strength - cand_strength) <= 1:
            strength_consistent += 1

    recall = len(matched) / len(baseline_signals)
    if not matched:
        return 0.0

    direction_match_rate = direction_matches / len(matched)
    strength_consistency_rate = strength_consistent / len(matched)
    return recall * direction_match_rate * strength_consistency_rate


def q3_strength_drift(previous: dict, candidate: dict) -> dict:
    candidate_by_category = {}
    for signal in candidate.get("signals", []):
        candidate_by_category.setdefault(signal.get("category"), []).append(signal)

    upgrades = 0
    downgrades = 0
    for prev_signal in previous.get("signals", []):
        category = prev_signal.get("category")
        candidates = candidate_by_category.get(category, [])
        if not candidates:
            continue
        cand_signal = candidates.pop(0)
        try:
            prev_strength = int(prev_signal.get("strength"))
            cand_strength = int(cand_signal.get("strength"))
        except (TypeError, ValueError):
            continue
        if cand_strength > prev_strength:
            upgrades += 1
        elif cand_strength < prev_strength:
            downgrades += 1

    return {
        "strength_upgrades": upgrades,
        "strength_downgrades": downgrades,
        "strength_net_shift": upgrades - downgrades,
    }


MONTH_PATTERN = (
    r"(?<![А-Яа-яЁё])"
    r"(?:январ[ьяе]?|феврал[ьяе]?|март(?:а|е)?|апрел[ьяе]?|"
    r"май|мая|мае|июн[ьяе]?|июл[ьяе]?|август[ае]?|"
    r"сентябр[ьяе]?|октябр[ьяе]?|ноябр[ьяе]?|декабр[ьяе]?)"
    r"(?![А-Яа-яЁё])"
)


RESIDUAL_FINGERPRINT_PATTERNS = {
    "hard_fail": [
        MONTH_PATTERN,
        r"\b20\d{2}\b|\b19\d{2}\b",
        r"COVID|ковид|коронавирус|пандем",
        r"Citi|Пят[её]роч|ФРГ|SWIFT|санкц|контрсанкц|спецоперац|военн\w+ операц|Украин|Донбасс",
        r"Brent|WTI|Urals",
    ],
    "warn": [
        r"историческ\w+ максимум",
        r"годов\w+ минимум",
        r"стагфляц",
        r"голландск\w+ болезн",
        r"сырьев\w+ суперцикл",
        r"за три квартала",
        r"овсян\w+ хлоп",
        r"бюджетн\w+ правил",
        r"резервн\w+ валют",
        r"газ\w* в Европе.*ключев\w+ триггер",
        r"посткризисн\w+ волатильност",
        r"глава центробанка.*сбережен",
        r"хранит сбережен",
        r"заморозк\w+ резерв",
        r"капитал\w+ контрол",
        r"параллельн\w+ импорт",
        r"недружественн\w+ стран",
    ],
}


def residual_fingerprint_check(text: str, patterns: dict[str, list[str]]) -> dict:
    result = {"hard_fail": [], "warn": [], "manual_fail": False}
    for severity in ("hard_fail", "warn"):
        for pattern in patterns[severity]:
            for match in re.finditer(pattern, text, flags=re.IGNORECASE):
                result[severity].append(
                    {
                        "pattern": pattern,
                        "match": match.group(0),
                        "span": [match.start(), match.end()],
                    }
                )
    result["manual_fail"] = bool(result["hard_fail"])
    return result


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


## Cycle Runner

In [3]:
def initial_summary_for_point(point: dict) -> str:
    if point["point_label"] == "calm_2021":
        if not CALM_2021_RAW_SOURCE.exists():
            raise FileNotFoundError(f"Frozen calm_2021 raw summary not found: {CALM_2021_RAW_SOURCE}")
        return CALM_2021_RAW_SOURCE.read_text(encoding="utf-8")
    return load_summary(DB_PATH, point["run_date"])


def benchmark_record_metadata() -> dict:
    return {
        "benchmark_mode": BENCHMARK_MODE,
        "run_label": RUN_LABEL,
        "model_preset_description": MODEL_PRESET_DESCRIPTION,
        "out_root": str(OUT_ROOT),
        "model_roles": {"N": MODEL_N, "J1": MODEL_J1},
    }


def run_cycle(point: dict) -> dict:
    point_label = point["point_label"]
    run_date = point["run_date"]
    out_dir = point["out_dir"]
    out_dir.mkdir(parents=True, exist_ok=True)

    raw_summary = initial_summary_for_point(point)
    raw_summary_path = out_dir / "iter_00_raw.md"
    raw_summary_path.write_text(raw_summary, encoding="utf-8")

    q3_baseline_text = call_llm(
        model=MODEL_J1,
        system_prompt=Q3_SIGNALS_SYSTEM,
        user_prompt=f"Summary:\n\n{raw_summary}",
        temperature=TEMPERATURE_J1,
    )
    q3_baseline = extract_json(q3_baseline_text)
    q3_baseline_path = out_dir / "iter_00_q3_baseline.json"
    write_json(q3_baseline_path, q3_baseline)

    baseline_record = {
        "point_label": point_label,
        "run_date": run_date,
        **benchmark_record_metadata(),
        "iteration": 0,
        "raw_summary_path": str(raw_summary_path),
        "q3_baseline_path": str(q3_baseline_path),
        "status": "baseline",
    }
    write_json(out_dir / "iter_00_record.json", baseline_record)

    accepted_iteration = 0
    accepted_summary = raw_summary
    accepted_summary_path = raw_summary_path
    accepted_q3 = q3_baseline
    accepted_q3_path = q3_baseline_path
    accepted_residual = residual_fingerprint_check(raw_summary, RESIDUAL_FINGERPRINT_PATTERNS)
    stop_status = None
    rollback_from_iteration = None
    stopping_iteration = None
    stopping_iteration_record_path = None
    stopping_iteration_residual_check_path = None
    stopping_iteration_residual = None

    print(f"Baseline saved for {point_label}: {raw_summary_path}")

    for iteration in range(1, N_MAX + 1):
        prefix = f"iter_{iteration:02d}"
        prev_summary = accepted_summary
        prev_summary_path = accepted_summary_path
        prev_q3 = accepted_q3
        prev_q3_path = accepted_q3_path

        q1_text = call_llm(
            model=MODEL_J1,
            system_prompt=Q1_EVIDENCE_SYSTEM,
            user_prompt=f"Summary:\n\n{prev_summary}",
            temperature=TEMPERATURE_J1,
        )
        q1_identifiers = extract_json(q1_text)
        q1_path = out_dir / f"{prefix}_q1_identifiers.json"
        write_json(q1_path, q1_identifiers)

        rewriter_user_prompt = "\n\n".join(
            [
                "Previous summary:",
                prev_summary,
                "J1 Q1-evidence identifiers to mask:",
                json.dumps(q1_identifiers.get("identifiers", []), ensure_ascii=False, indent=2),
                "J1 Q3 signals_to_preserve:",
                json.dumps(prev_q3.get("signals", []), ensure_ascii=False, indent=2),
            ]
        )
        candidate_summary = call_llm(
            model=MODEL_N,
            system_prompt=N_REWRITER_SYSTEM,
            user_prompt=rewriter_user_prompt,
            temperature=TEMPERATURE_N,
        )
        candidate_path = out_dir / f"{prefix}_candidate.md"
        candidate_path.write_text(candidate_summary, encoding="utf-8")

        q3_candidate_text = call_llm(
            model=MODEL_J1,
            system_prompt=Q3_SIGNALS_SYSTEM,
            user_prompt=f"Summary:\n\n{candidate_summary}",
            temperature=TEMPERATURE_J1,
        )
        q3_candidate = extract_json(q3_candidate_text)
        q3_candidate_path = out_dir / f"{prefix}_q3_candidate.json"
        write_json(q3_candidate_path, q3_candidate)

        q3_preservation = q3_preservation_score(prev_q3, q3_candidate)
        q3_drift = q3_strength_drift(prev_q3, q3_candidate)
        residual = residual_fingerprint_check(candidate_summary, RESIDUAL_FINGERPRINT_PATTERNS)
        residual_check_path = out_dir / f"iter_{iteration:02d}_residual_check.json"
        write_json(residual_check_path, residual)
        length_ratio = len(candidate_summary) / len(prev_summary) if prev_summary else 0.0

        status = "continue"
        rollback = False
        if q3_preservation < Y_FLOOR:
            status = "signal_collapse"
            rollback = True
        elif residual["manual_fail"]:
            status = "residual_hard_fail"
            rollback = ROLLBACK_ON_HARD_RESIDUAL
        elif MANUAL_STOP:
            status = "manual_review"
        elif iteration >= N_MAX:
            status = "hard_capped"

        iteration_record = {
            "point_label": point_label,
            "run_date": run_date,
            **benchmark_record_metadata(),
            "iteration": iteration,
            "prev_summary_path": str(prev_summary_path),
            "candidate_path": str(candidate_path),
            "q1_identifiers_path": str(q1_path),
            "q3_prev_path": str(prev_q3_path),
            "q3_candidate_path": str(q3_candidate_path),
            "q3_preservation": q3_preservation,
            "q3_strength_upgrades": q3_drift["strength_upgrades"],
            "q3_strength_downgrades": q3_drift["strength_downgrades"],
            "q3_strength_net_shift": q3_drift["strength_net_shift"],
            "residual_check_path": str(residual_check_path),
            "residual_hard_fail_count": len(residual["hard_fail"]),
            "residual_warn_count": len(residual["warn"]),
            "residual_manual_fail": residual["manual_fail"],
            "length_ratio": length_ratio,
            "status": status,
        }
        iteration_record_path = out_dir / f"{prefix}_record.json"
        write_json(iteration_record_path, iteration_record)

        print(
            f"{point_label} {prefix}: status={status}, "
            f"q3={q3_preservation:.3f}, hard={len(residual['hard_fail'])}, warn={len(residual['warn'])}"
        )

        if status == "continue":
            accepted_iteration = iteration
            accepted_summary = candidate_summary
            accepted_summary_path = candidate_path
            accepted_q3 = q3_candidate
            accepted_q3_path = q3_candidate_path
            accepted_residual = residual
            continue

        stop_status = status
        stopping_iteration = iteration
        stopping_iteration_record_path = iteration_record_path
        stopping_iteration_residual_check_path = residual_check_path
        stopping_iteration_residual = residual
        if rollback:
            rollback_from_iteration = iteration
        else:
            accepted_iteration = iteration
            accepted_summary = candidate_summary
            accepted_summary_path = candidate_path
            accepted_q3 = q3_candidate
            accepted_q3_path = q3_candidate_path
            accepted_residual = residual
        break

    if stop_status is None:
        stop_status = "hard_capped"
        stopping_iteration = N_MAX

    final_q3_drift = q3_strength_drift(q3_baseline, accepted_q3)
    final_record = {
        "point_label": point_label,
        "run_date": run_date,
        **benchmark_record_metadata(),
        "final_iteration": accepted_iteration,
        "final_status": stop_status,
        "final_candidate_path": str(accepted_summary_path),
        "rollback_from_iteration": rollback_from_iteration,
        "stopping_iteration": stopping_iteration,
        "stopping_iteration_record_path": str(stopping_iteration_record_path) if stopping_iteration_record_path else None,
        "stopping_iteration_residual_check_path": str(stopping_iteration_residual_check_path) if stopping_iteration_residual_check_path else None,
        "stopping_iteration_residual_hard_fail_count": len(stopping_iteration_residual["hard_fail"]) if stopping_iteration_residual else None,
        "stopping_iteration_residual_warn_count": len(stopping_iteration_residual["warn"]) if stopping_iteration_residual else None,
        "final_candidate_residual_hard_fail_count": len(accepted_residual["hard_fail"]) if accepted_residual else None,
        "final_candidate_residual_warn_count": len(accepted_residual["warn"]) if accepted_residual else None,
        "final_q3_preservation": q3_preservation_score(q3_baseline, accepted_q3),
        "length_ratio": len(accepted_summary) / len(raw_summary) if raw_summary else 0.0,
        "q3_strength_net_shift": final_q3_drift["strength_net_shift"],
    }
    write_json(out_dir / "final_record.json", final_record)
    print(f"Final record saved for {point_label}: {out_dir / 'final_record.json'}")
    return final_record


## Run benchmark points


In [4]:
final_records = []
for point in ACTIVE_POINTS:
    final_records.append(run_cycle(point))

final_records


Baseline saved for shock_war: data/runs/e06_cycle_model_bench/sonnet46__qwen397b/shock_war/iter_00_raw.md
shock_war iter_01: status=residual_hard_fail, q3=0.833, hard=3, warn=0
Final record saved for shock_war: data/runs/e06_cycle_model_bench/sonnet46__qwen397b/shock_war/final_record.json


[{'point_label': 'shock_war',
  'run_date': '2022-03-25',
  'benchmark_mode': True,
  'run_label': 'sonnet46__qwen397b',
  'model_preset_description': 'Strong Claude N candidate + same large Qwen J1; clean N-only benchmark',
  'out_root': 'data/runs/e06_cycle_model_bench/sonnet46__qwen397b',
  'model_roles': {'N': 'anthropic/claude-sonnet-4.6',
   'J1': 'qwen/qwen3.5-397b-a17b'},
  'final_iteration': 0,
  'final_status': 'residual_hard_fail',
  'final_candidate_path': 'data/runs/e06_cycle_model_bench/sonnet46__qwen397b/shock_war/iter_00_raw.md',
  'rollback_from_iteration': 1,
  'stopping_iteration': 1,
  'stopping_iteration_record_path': 'data/runs/e06_cycle_model_bench/sonnet46__qwen397b/shock_war/iter_01_record.json',
  'stopping_iteration_residual_check_path': 'data/runs/e06_cycle_model_bench/sonnet46__qwen397b/shock_war/iter_01_residual_check.json',
  'stopping_iteration_residual_hard_fail_count': 3,
  'stopping_iteration_residual_warn_count': 0,
  'final_candidate_residual_hard_f

## Summary table

In [5]:
rows = []
summary_points = []
for point in ACTIVE_POINTS:
    final_record_path = point["out_dir"] / "final_record.json"
    record = json.loads(final_record_path.read_text(encoding="utf-8"))
    summary_points.append(
        {
            "point_label": record["point_label"],
            "final_status": record["final_status"],
            "final_iteration": record["final_iteration"],
            "rollback_from_iteration": record["rollback_from_iteration"],
            "final_q3_preservation": record["final_q3_preservation"],
            "length_ratio": record["length_ratio"],
            "final_candidate_residual_hard_fail_count": record["final_candidate_residual_hard_fail_count"],
            "final_candidate_residual_warn_count": record["final_candidate_residual_warn_count"],
            "stopping_iteration_residual_hard_fail_count": record["stopping_iteration_residual_hard_fail_count"],
            "stopping_iteration_residual_warn_count": record["stopping_iteration_residual_warn_count"],
            "q3_strength_net_shift": record["q3_strength_net_shift"],
        }
    )
    rows.append(
        {
            "run_label": record["run_label"],
            "model_N": record["model_roles"]["N"],
            "model_J1": record["model_roles"]["J1"],
            "model_preset_description": record["model_preset_description"],
            "out_root": record["out_root"],
            "point_label": record["point_label"],
            "final_status": record["final_status"],
            "final_iteration": record["final_iteration"],
            "rollback_from_iteration": record["rollback_from_iteration"],
            "stopping_iteration": record["stopping_iteration"],
            "final_q3_preservation": round(record["final_q3_preservation"], 3),
            "length_ratio": round(record["length_ratio"], 3),
            "final_candidate_residual_hard_fail_count": record["final_candidate_residual_hard_fail_count"],
            "final_candidate_residual_warn_count": record["final_candidate_residual_warn_count"],
            "stopping_iteration_residual_hard_fail_count": record["stopping_iteration_residual_hard_fail_count"],
            "stopping_iteration_residual_warn_count": record["stopping_iteration_residual_warn_count"],
            "q3_strength_net_shift": record["q3_strength_net_shift"],
        }
    )

benchmark_summary = {
    "benchmark_mode": BENCHMARK_MODE,
    "run_label": RUN_LABEL,
    "model_preset_description": MODEL_PRESET_DESCRIPTION,
    "out_root": str(OUT_ROOT),
    "model_roles": {"N": MODEL_N, "J1": MODEL_J1},
    "point_filter": POINT_FILTER,
    "points": summary_points,
}
OUT_ROOT.mkdir(parents=True, exist_ok=True)
write_json(OUT_ROOT / "benchmark_summary.json", benchmark_summary)

headers = [
    "run_label",
    "model_N",
    "model_J1",
    "model_preset_description",
    "out_root",
    "point_label",
    "final_status",
    "final_iteration",
    "rollback_from_iteration",
    "stopping_iteration",
    "final_q3_preservation",
    "length_ratio",
    "final_candidate_residual_hard_fail_count",
    "final_candidate_residual_warn_count",
    "stopping_iteration_residual_hard_fail_count",
    "stopping_iteration_residual_warn_count",
    "q3_strength_net_shift",
]

try:
    from tabulate import tabulate
except ImportError:
    tabulate = None

if tabulate:
    print(tabulate(rows, headers="keys", tablefmt="github"))
else:
    widths = {header: len(header) for header in headers}
    for row in rows:
        for header in headers:
            widths[header] = max(widths[header], len(str(row[header])))

    print(" | ".join(header.ljust(widths[header]) for header in headers))
    print(" | ".join("-" * widths[header] for header in headers))
    for row in rows:
        print(" | ".join(str(row[header]).ljust(widths[header]) for header in headers))


run_label          | model_N                     | model_J1               | model_preset_description                                               | out_root                                           | point_label | final_status       | final_iteration | rollback_from_iteration | stopping_iteration | final_q3_preservation | length_ratio | final_candidate_residual_hard_fail_count | final_candidate_residual_warn_count | stopping_iteration_residual_hard_fail_count | stopping_iteration_residual_warn_count | q3_strength_net_shift
------------------ | --------------------------- | ---------------------- | ---------------------------------------------------------------------- | -------------------------------------------------- | ----------- | ------------------ | --------------- | ----------------------- | ------------------ | --------------------- | ------------ | ---------------------------------------- | ----------------------------------- | ------------------------------------------- | -